In [7]:
import pandas as pd

In [8]:
# Step 0: Load file (only necessary columns + optimized dtype, to save memory)
INPUT_FILE = "dataset_accepted_loan.csv"  

usecols = [
    "id",                    
    "member_id",             
    "annual_inc",
    "emp_length",
    "home_ownership",
    "addr_state",
    "loan_amnt",
    "term",
    "int_rate",
    "grade",
    "sub_grade",
    "purpose",
    "issue_d",
    "installment",
    "fico_range_low",
    "fico_range_high",
    "delinq_2yrs",
    "open_acc",
    "pub_rec",
    "revol_util",
    "dti",
    "loan_status",
    "last_pymnt_d",
    "total_pymnt",
    
    "verification_status",   
    "application_type",      
    "revol_bal",              
    "total_acc",              
    "mort_acc",                
    "pub_rec_bankruptcies",   
    "inq_last_6mths",          
]

dtype_map = {
    "loan_amnt": "float32",
    "int_rate": "float32",
    "installment": "float32",
    "annual_inc": "float32",
    "dti": "float32",
    "fico_range_low": "float32",
    "fico_range_high": "float32",
    "delinq_2yrs": "float32",
    "open_acc": "float32",
    "pub_rec": "float32",
    "total_pymnt": "float32",
    "revol_bal": "float32",
    "total_acc": "float32",
    "mort_acc": "float32",
    "pub_rec_bankruptcies": "float32",
    "inq_last_6mths": "float32",
}

df = pd.read_csv(INPUT_FILE, usecols=usecols, dtype=dtype_map, low_memory=False)

print(f"Total rows loaded (before excluding footer/garbage rows): {len(df):,}")

Total rows loaded (before excluding footer/garbage rows): 2,260,701


In [9]:

# basic cleaning

df["id"] = pd.to_numeric(df["id"], errors="coerce")

rows_before = len(df)
df = df.dropna(subset=["id", "loan_status"])
rows_dropped = rows_before - len(df)
print(f"Excluded due to non-numeric/empty id or loan_status: {rows_dropped:,} row")

df["id"] = df["id"].astype("int64")

# member_id is usually left blank for privacy reasons — make sure
member_id_filled = df["member_id"].notna().sum()
print(f"Row number filled in member_id: {member_id_filled:,} (Usually it is 0.)")

Excluded due to non-numeric/empty id or loan_status: 33 row
Row number filled in member_id: 0 (Usually it is 0.)


In [10]:
#  Create surrogate borrower_id

# There is no way to track whether a person actually has multiple loans in this dataset
# (since member_id is blank). So we are assuming each loan is a borrower
# and using loan_id as the borrower_id (1 loan = 1 borrower surrogate key).
df["borrower_id"] = df["id"]

In [11]:

# Divide into four tables.


# --- Table 1: borrowers ---
borrowers = df[[
    "borrower_id", "annual_inc", "emp_length", "home_ownership", "addr_state", "dti",
    "verification_status", "application_type",
]].rename(columns={
    "annual_inc": "annual_income",
    "addr_state": "state",
})

# --- Table 2: loans ---
loans = df[[
    "id", "borrower_id", "loan_amnt", "term", "int_rate",
    "installment", "grade", "sub_grade", "purpose", "issue_d"
]].rename(columns={"id": "loan_id"})

# --- Table 3: credit_history ---
credit_history = df[[
    "borrower_id", "fico_range_low", "fico_range_high",
    "delinq_2yrs", "open_acc", "pub_rec", "revol_util",
    "revol_bal", "total_acc", "mort_acc", "pub_rec_bankruptcies", "inq_last_6mths",
]]

# --- Table 4: payment_status ---
payment_status = df[[
    "id", "loan_status", "last_pymnt_d", "total_pymnt"
]].rename(columns={"id": "loan_id"})

In [12]:
#  Save as CSV (import these into MySQL/PostgreSQL later)
borrowers.to_csv("borrowers.csv", index=False)
loans.to_csv("loans.csv", index=False)
credit_history.to_csv("credit_history.csv", index=False)
payment_status.to_csv("payment_status.csv", index=False)

print("4 tables have been prepared.:")
print(f"  borrowers.csv       -> {len(borrowers):,} row, {borrowers.shape[1]} column")
print(f"  loans.csv           -> {len(loans):,} row, {loans.shape[1]} column")
print(f"  credit_history.csv  -> {len(credit_history):,} row, {credit_history.shape[1]} column")
print(f"  payment_status.csv  -> {len(payment_status):,} row, {payment_status.shape[1]} column")

4 tables have been prepared.:
  borrowers.csv       -> 2,260,668 row, 8 column
  loans.csv           -> 2,260,668 row, 10 column
  credit_history.csv  -> 2,260,668 row, 12 column
  payment_status.csv  -> 2,260,668 row, 4 column
